(collections-section)=

# Collection Families and Selection


```{contents}
:local:
:depth: 2
```


```{index} collections
```



```{index} Collections
```

<span id="data-collections"></span>


A bookstore handles many groups of values: the quantities sold during a week, the items in a customer's basket, stock quantities indexed by product code, and orders waiting for collection. Each group contains several values, but the program does different work with each one. Finding Tuesday's sales, looking up a product code, and selecting the next order are different questions. The way you organize the values should support the question you need to answer.

A **collection** groups values and provides operations for working with them. Arrays are the first collection type we studied: {doc}`Chapter 7 <../07_arrays/0700_arrays>` introduced fixed-length, indexed sequences. Keeping that chapter separate gave us time to practice indexing, traversal, and multidimensional data. It did not put arrays outside the collection category. This chapter broadens our choices to other ready-made collection types in the .NET standard library. These classes are often called **built-in collections** because they come with .NET, although they are library types used by C#, rather than separate C# language features. Arrays have direct C# language syntax, such as `int[]`; collection classes such as `List<int>` are library types. Both belong in our collection-family comparison. The runtime agrees. A one-dimensional array implements the same collection interfaces as `List<T>`, including `IEnumerable<T>`, `ICollection<T>`, and `IList<T>`, which is why `foreach` works on an array and why a method that accepts a collection accepts an array.[^0801_collections-array-interfaces] What an array cannot do is change its length; {ref}`a run-time check <array-fixed-size-collection>` in the Collection Operations section shows the difference.

This section introduces the big ideas before the detailed syntax: what a collection promises, how values can be accessed, and how to choose among collection families. You will see eight useful families and small demonstrations of their behavior. The next two sections develop `List<T>` and `Dictionary<TKey,TValue>` in depth. For the other families, your goal here is to recognize the need they address and predict their basic behavior, not to memorize every method or implement a data structure.


**Reading path:** First explain the access question, compare the families, and connect their operations. Work through arrays/lists, dictionaries, and shared references before the order-summary exercise. Use the linked-list, set, stack/queue, and priority-queue demonstrations to recognize their policies; their implementations and detailed APIs belong to later chapters. The operation tables are references to revisit, not a list of signatures to memorize in one sitting.


```{index} Collections and Operations
```

## Collections and Operations




```{index} collection; element; collection operation; data structure
```


An **element** is one item stored in a collection. An array of sales quantities has numbers as its elements; a list of product codes has strings. A dictionary stores entries that associate a key with a value. The surrounding collection gives these items an organization and a set of supported actions.

Several operations recur across collection families:

| Operation | Question it answers | Bookstore example |
| --- | --- | --- |
| Add or insert | How does a new item enter the group? | Add a product code to the current basket. |
| Retrieve | How do you obtain a particular item? | Find the stock quantity associated with a product code. |
| Update | How does stored information change? | Replace a recorded quantity after a stock check. |
| Remove | How does an item leave the group? | Remove a canceled item from a basket. |
| Membership test | Is this item or key present? | Check whether a customer has already used a promotion. |
| Traverse | How do you visit the stored items? | Print every item on an order. |
| Inspect size | How many items are stored? | Count the orders waiting for collection. |

Not every collection offers every operation, and the method names differ. A stack deliberately limits which item you retrieve next. An array lets you replace an element but does not let you add a new position to that same array. The operations and their rules form a collection's **contract**: the behavior a programmer can rely on.

A **data structure** is an organization of stored information that supports operations. A library collection gives you an already implemented data structure through a public interface of methods and properties. You can use a list before you know how its storage grows, just as you can call `Math.Round` before implementing a rounding algorithm. Later chapters examine the implementation choices behind these operations.

Keep the question of behavior separate from the question of storage. A queue promises a first-in, first-out processing rule. That rule does not require you to know whether a particular implementation uses an array or linked nodes. Conversely, the fact that two collections both store strings does not mean that their access rules are interchangeable.


```{index} Collection Families
```

(collections-by-access-pattern)=

## Collection Families




```{index} collection; families; access pattern
```


An **access pattern** describes how a program usually retrieves and processes its data. Do you know an item's position? Do you know a key that identifies it? Do you only need to know whether it is present? Must items be processed in arrival order?

In the type names below, `T` stands for an element type you choose, such as `string` or `int`. `TKey`, `TValue`, and `TPriority` name the corresponding key, value, and priority types. The examples later in this section use `new` to construct a collection and dot notation to call its operations. First, read the table for the organizing idea behind each family; the next discussion explains generic type arguments before you examine the individual examples.

**Table: An introductory map of .NET collection families.** Adapted from Microsoft's collection-selection guidance, with original business examples and an added priority-queue entry.[^0801_collections-collection-guide][^0801_collections-priority-queue] These eight families are a teaching selection, not Microsoft's exhaustive classification. Sorted variants and collections for specialized settings extend this map.

| Family | Representative type | Main access rule | Business use |
| --- | --- | --- | --- |
| Array | `T[]`, such as `int[]` | Retrieve by index; length is fixed. | Store one quantity for each of five business days. |
| List | `List<T>` | Retrieve by index; the sequence can grow or shrink. | Build a customer's basket as items are selected. |
| Linked list | `LinkedList<T>` | Follow linked nodes; insert or remove at a known node. | Insert a processing step beside an already located step. |
| Dictionary | `Dictionary<TKey,TValue>` | Retrieve a value using its unique key. | Look up stock by product code. |
| Set | `HashSet<T>` | Test membership; equal values occur only once. | Track distinct customers who used a promotion. |
| Stack | `Stack<T>` | Retrieve the most recently added item first. | Keep a history of actions to undo. |
| Queue | `Queue<T>` | Retrieve the earliest added item first. | Process service requests in arrival order. |
| Priority queue | `PriorityQueue<TElement,TPriority>` | Retrieve an item with the lowest priority value first. | Process urgent requests before routine requests. |

The same business process can need several collections. A bookstore might use a dictionary for stock lookup, a list for basket contents, and a queue for orders awaiting service. Choosing one collection does not commit the whole application to that family.

Three distinctions help you read the table. **Position** identifies where an element currently sits in a sequence. **Identity** identifies which entity you mean, such as a product code. **Processing order** determines which waiting item is handled next. Confusing these ideas produces fragile programs: a product code should not silently change because an earlier item was removed from a list.


```{index} Choosing a Collection
```

## Choosing a Collection




```{index} collection selection; duplicate; order; lookup
```


Start with the work the program must do, rather than the longest list of available methods. For a bookstore promotion, suppose the manager asks for four results: the order of product selections, the current stock for a known code, the distinct customers who participated, and the next request to process.

An ordered list preserves selections, including repeated product codes. A dictionary associates a product code with its stock quantity. A set records distinct participating customers. A queue selects the next request if the policy is arrival order; a priority queue is appropriate if a defined priority rule should determine service. These structures can collaborate in one application because they answer different questions.

Use the following questions to explain a choice:

1. **What is one element?** A product code, an integer quantity, or a request is a different unit of information. Define what an entry represents before choosing a type.
2. **How will you retrieve it?** Distinguish an index, an identifying key, a membership test, and a next-item rule.
3. **Do order and duplicates carry information?** Removing duplicates from a transaction history can lose events. Preserving every duplicate in a distinct-customer report can misstate participation.
4. **Does the size change?** Decide whether fixed positions describe the task or whether adding and removing elements is normal work.
5. **What operations must be safe?** Consider missing keys, empty collections, invalid values, and whether another part of the program shares the object.

Performance matters too, but "fast" needs an operation and a situation. A dictionary is designed for lookup by key; it does not make every kind of search or report automatically fast. A linked list can insert beside a known node without shifting a suffix, but finding that node can take traversal. Later chapters develop these costs systematically. Here, explain the behavioral fit first and avoid claims that one family is always best.

Trees and graphs extend the larger study of data structures. They model hierarchies and networks, rather than adding two more standard generic classes named `Tree<T>` and `Graph<T>` to this table. Chapters 17 and 20 teach those structures and their applications. Arrays are developed in Chapter 7; linked structures in Chapter 14; stacks and queues in Chapter 15; priority queues in Chapter 18; and hashing and sets in Chapter 19.

For this chapter's detailed coding and assignments, concentrate on lists and dictionaries. The broader family map should help you recognize alternatives and ask better questions about the data. You do not need the internal mechanics of a hash table or heap to explain why a lookup table and a waiting line have different access rules.


**Guided choice: the daily order desk**

A shop stores five daily sales totals, receives a changing sequence of basket items, looks up stock by product code, and handles requests in arrival order. For each requirement, name the identifying information (position, key, or processing order), select a family, and explain one unsuitable alternative. Then change the requirement: what if urgent requests should go first?


```{dropdown} Reasoning to compare with yours
Five fixed positions fit an array. Basket items fit a list because their number changes and repeated selections can matter. Stock lookup fits a dictionary keyed by product code. Arrival-order service fits a queue; a stack would serve the most recent request first. Urgency suggests a priority queue with an explicitly defined priority policy. These choices follow the operations, not the fact that all four groups contain business data.
```


The order desk reduces to three access questions. Each one points to a collection family and to the operation that answers it.

| Question the program must answer | Collection family | Operation |
| --- | --- | --- |
| **Position:** what was sold on Tuesday, the second day of the week? | Array or list | `dailySales[1]` |
| **Key:** how many of product `PEN` are in stock? | Dictionary | `stock["PEN"]` |
| **Next:** which service request arrived first? | Queue | `requests.Dequeue()` |

One application can use all three. The stored type alone does not choose the collection, and the table compares access policies, not performance.


```{rubric} Footnotes
```


[^0801_collections-collection-guide]: Microsoft. [*Collections and Data Structures*](https://learn.microsoft.com/en-us/dotnet/standard/collections/) and [*Selecting a Collection Class*](https://learn.microsoft.com/en-us/dotnet/standard/collections/selecting-a-collection-class). Microsoft Learn. Accessed October 16, 2026. The table adapts the documentation's operation-based selection approach; its eight-family grouping and bookstore examples are the author's teaching organization. The priority-queue entry is additionally supported by the API reference below.
[^0801_collections-linked-list]: The .NET [LinkedList<T> reference](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.linkedlist-1?view=net-10.0) describes its node-based, doubly linked organization.
[^0801_collections-priority-queue]: The .NET [PriorityQueue<TElement,TPriority> reference](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.priorityqueue-2?view=net-10.0) specifies lowest-priority removal and does not guarantee FIFO behavior for equal priorities.
[^0801_collections-collection-interface]: `IEnumerable<T>` supplies an enumerator used by `foreach`. Enumeration order belongs to each type's contract; supporting enumeration does not imply sorted, insertion, or priority order.

[^0801_collections-collection-operations]: Microsoft Learn, .NET 10 API reference: [List<T>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.list-1?view=net-10.0), [LinkedList<T>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.linkedlist-1?view=net-10.0), [Dictionary<TKey,TValue>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.dictionary-2?view=net-10.0), [HashSet<T>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.hashset-1?view=net-10.0), [Stack<T>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.stack-1?view=net-10.0), [Queue<T>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.queue-1?view=net-10.0), and [PriorityQueue<TElement,TPriority>](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.priorityqueue-2?view=net-10.0). This table compares selected operations rather than listing each type's complete API.
[^0801_collections-array-interfaces]: The .NET [Array class reference](https://learn.microsoft.com/en-us/dotnet/api/system.array) lists the interfaces arrays implement. Single-dimensional, zero-based arrays also implement the generic `IList<T>`, `ICollection<T>`, `IEnumerable<T>`, `IReadOnlyList<T>`, and `IReadOnlyCollection<T>`; because the size is fixed, their `Add` and `Remove` throw `NotSupportedException`.
